In [13]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "volter2016great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Voelter_2016_exp1_Cognit_GRON.csv")
complete_path_2 = os.path.join(original_data_pathway, "Voelter_2016_exp2_Cognit_GRON.csv")
complete_path_3 = os.path.join(original_data_pathway, "Voelter_2016_exp3_Cognit_GRON.csv")
complete_path_4 = os.path.join(original_data_pathway, "Voelter_2016_exp4_Cognit_GRON.csv")
complete_path_5 = os.path.join(original_data_pathway, "Voelter_2016_SM_exp5_Cognit_GRON.csv")
complete_path_6 = os.path.join(original_data_pathway, "volter2016great_exp5.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [14]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df1['experiment']="1"
df2 = pd.read_csv(complete_path_2)
df2['experiment']="2"
df3 = pd.read_csv(complete_path_3)
df3['experiment']="3"
df3.rename(columns={"chosen_side_first_response": "first_response_side"}, inplace=True)

In [15]:
df4 = pd.read_csv(complete_path_4)
df4['experiment']="4"
df4.rename(columns={"chosen_side_first_response": "first_response_side",
                    'chosen_side_second_response':'second_response_side'}, inplace=True)

In [16]:
df5 = pd.read_csv(complete_path_5)
df5['experiment']="5"
df5['publication_status']="published"
df5.rename(columns={"chosen_side_first_response": "first_response_side",
                    'chosen_side_second_response':'second_response_side'}, inplace=True)

In [17]:

df6 = pd.read_csv(complete_path_6, encoding='utf-8')
df6['experiment']="5"
df6['publication_status']="unpublished"



In [18]:
df6['blicket_object'].replace(',', '_', inplace=True, regex=True)
# df6['blicket_object'].unique()

In [19]:
df4.rename(columns={"Age": "age_temp",
    "Sex":"sex_temp"}, inplace=True)

replace_list_temp = [['sex_temp', 'm'],
                    ['sex_temp', 'f'],
                    ['age_temp', 'm'],
                    ['age_temp', 'f']]
for x,y in replace_list_temp:
    df4[x].replace(y, np.nan, inplace=True, regex=True)

In [20]:
spe_2=[] 
for index, row in df4.iterrows():
    if not pd.isna(row['sex_temp']):
        spe_2.append(row['sex_temp'])
    else:
        spe_2.append(row['age_temp'])
df4 = df4.assign(age=spe_2)

In [21]:
data_frames=[df1, df2, df3, df4, df5, df6]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "ape", 
        "species":"species_original",
        "sex":"sex_original",
        "trial_number":"trial",
        "group":"group_original"}, inplace=True)
    x['study_id']="volter2016great"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

fulldf = fulldf[fulldf.species_original!='human']

update_side_list = ['blicket_side', 'first_response_side', 'second_response_side','ac_side','rational_side']
fulldf = fulldf.copy()
for x in update_side_list:
    fulldf[x].replace('l', 'left', inplace=True)
    fulldf[x].replace('r', 'right', inplace=True)


# fulldf['first_response_side'].unique()


In [22]:
# fulldf.columns
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

fulldf.rename(columns={"ape": "participant", "age":"age_in_years", 'age_exact':"age_in_years"}, inplace=True)


In [23]:
fulldf=fulldf[[ 'study_id', 'experiment', 'publication_status', 'participant', 'age_in_years', 'sex', 'species',
        'phase','session', 'trial', 
        'condition',
          'order_condition','instruction',
        'demonstration', 'order_demo','order_a', 'blicket', 'blicket_side',
       'first_response', 'first_response_side', 
         'first_response_a_or_b',
       '100percent_object_chosen',  
       'blicket_object', 'first_response_100percent_object',
        'first_response_correct',
       'second_response', 'second_response_side', 'ac_side',
      #  'rational_side',
         'second_response_correct']]


In [24]:
fulldf['experiment'].unique()
fulldf.loc[fulldf.experiment == '5', ['condition']] = np.nan

In [25]:
for index in range(1,6):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'volter2016great_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'volter2016great_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

In [26]:
# obs1 = fulldf[fulldf['experiment'] == '0']
# obs1 = obs1.dropna(axis=1, how='all')

# comp_out_path_stand = os.path.join(out_pathway, 'volter2016great_training_standardized.csv')
# obs1.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

# names = obs1.columns.tolist()
# df = pd.DataFrame(names)
# df = df.rename(columns={0: "column_name"})
# df["description"] = ""
# volter2016great_training_glossary=df[["column_name", "description"]]

# comp_out_path_glossary = os.path.join(out_pathway, 'volter2016great_training_glossary.csv')
# volter2016great_training_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
